# Point Cloud Classification

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/opengeos/geoai/blob/main/docs/examples/point_cloud_classification.ipynb)

This notebook demonstrates how to classify 3D point clouds into semantic
categories (e.g., ground, buildings, vegetation) using RandLA-Net via the
`geoai.pointcloud` module.

The default model (`RandLANet_DALES`) is trained on the DALES aerial LiDAR
dataset and is suitable for airborne LiDAR data. Additional models are
available for USGS 3DEP aerial LiDAR (3DEP), street-level (SemanticKITTI),
urban (Toronto3D), and indoor (S3DIS) scenarios.

**Reference:**
Hu et al., "RandLA-Net: Efficient Semantic Segmentation of Large-Scale
Point Clouds," CVPR 2020.

## Install packages

In [ ]:
# %pip install "geoai-py[pointcloud]" leafmap[lidar]
# Note: Open3D 0.18+ requires PyTorch 2.2.x and NumPy < 2.
# On Google Colab, uncomment and run the following lines:
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
# !pip install "geoai-py[pointcloud]" leafmap[lidar]
# !pip install "numpy<2"  # MUST be last to avoid being overwritten
# After installing, restart the runtime before running the next cell.

## Import libraries

In [ ]:
import os

import leafmap
import numpy as np

from geoai.pointcloud import (
    ASPRS_CLASSES,
    PointCloudClassifier,
    classify_point_cloud,
    list_pointcloud_models,
)

## List available models

GeoAI ships pre-trained RandLA-Net checkpoints for five scenarios:
aerial LiDAR (DALES), USGS 3DEP aerial LiDAR (3DEP), outdoor driving
(SemanticKITTI), urban mapping (Toronto3D), and indoor scanning (S3DIS).

In [ ]:
models = list_pointcloud_models()
for name, desc in models.items():
    print(f"{name}:\n  {desc}\n")

## ASPRS classification codes

The standard LAS classification codes used in airborne LiDAR.

In [ ]:
for code, name in sorted(ASPRS_CLASSES.items()):
    print(f"  {code:3d}: {name}")

## Download sample data

Download a sample LiDAR point cloud for classification.

In [ ]:
url = "https://opengeos.org/data/lidar/madison.zip"
filename = "madison.las"
leafmap.download_file(url, "madison.zip", unzip=True)

## Visualize the input point cloud

Use leafmap to view the raw point cloud before classification.

In [ ]:
leafmap.view_lidar(filename, cmap="terrain", backend="pyvista")

## Initialize the classifier

Create a `PointCloudClassifier` with the DALES pre-trained model (default).
This model is trained on aerial LiDAR data and classifies into 9 categories:
ground, vegetation, buildings, cars, trucks, poles, power lines, fences,
and unknown. It uses only XYZ coordinates (no RGB required).

In [ ]:
classifier = PointCloudClassifier(
    model_name="RandLANet_DALES",
    device="cpu",  # Use "cuda" if you have a GPU
)

## Run classification

Classify the point cloud and save the result to a new LAS file.

In [ ]:
output_path = "madison_classified.las"
labels, probabilities = classifier.classify(
    filename,
    output_path=output_path,
)
print(f"Classified {len(labels)} points into {len(np.unique(labels))} classes.")

## Examine classification results

View summary statistics of the classified point cloud.

In [ ]:
stats = classifier.summary(output_path)
print(f"Total points: {stats['total_points']:,}")
print(f"\nClass distribution:")
for name, pct in stats["class_percentages"].items():
    count = stats["class_counts"][name]
    print(f"  {name:30s}: {count:>8,} ({pct:.1f}%)")

## Visualize classified point cloud

View the classified point cloud colored by class labels.

In [ ]:
classifier.visualize(output_path, backend="pyvista", cmap="tab20")

## Quick classification (convenience function)

For one-off classification, use the `classify_point_cloud()` convenience
function which creates the model and runs inference in a single call.

In [ ]:
# labels, probs = classify_point_cloud(
#     "input.las",
#     output_path="output_classified.las",
#     model_name="RandLANet_DALES",
# )

## Fine-tuning on custom data

If you have labeled LAS/LAZ files, you can fine-tune the model on your
own data.  Place training files in one directory and (optionally)
validation files in another.  Each file must have the `classification`
field populated with ground-truth labels.

In [ ]:
# Fine-tune on your own labeled LAS/LAZ files:
# history = classifier.train(
#     train_dir="path/to/train_data/",
#     val_dir="path/to/val_data/",
#     epochs=50,
#     learning_rate=0.001,
# )